<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/13-performance-tuning/02-bucketing-for-joins.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 13.2 — Bucketing: pay the shuffle once, not every run

Bucket two tables on the same join key via `saveAsTable`, confirm the
`Exchange` disappears from the join plan, then simulate "many runs"
by timing the same join repeatedly — bucketed vs not. Writes to a
local `spark-warehouse` next to the notebook (4.3's convention) and
self-cleans.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("13.2-bucketing-for-joins")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## Baseline: two large-ish tables, joined normally (shuffles every time)

In [ ]:
orders = (spark.range(2_000_000)
          .withColumn("customer_id", (col("id") % 5000).cast("int"))
          .withColumn("amount", col("id") % 500))
customers = (spark.range(5000)
             .withColumnRenamed("id", "customer_id")
             .withColumn("segment", (col("customer_id") % 4).cast("int")))

print("UNBUCKETED join plan -- expect Exchange on both sides:")
orders.join(customers, "customer_id").explain()

## Write both bucketed, same count, same key, via `saveAsTable`

Bucketing metadata lives in the metastore — `saveAsTable`, not
`.parquet()` (the lesson's non-negotiable requirement).

In [ ]:
import shutil
spark.sql("DROP TABLE IF EXISTS orders_bucketed")
spark.sql("DROP TABLE IF EXISTS customers_bucketed")

(orders.write.mode("overwrite")
 .bucketBy(8, "customer_id").sortBy("customer_id")
 .saveAsTable("orders_bucketed"))

(customers.write.mode("overwrite")
 .bucketBy(8, "customer_id").sortBy("customer_id")
 .saveAsTable("customers_bucketed"))

print("tables:", [t.name for t in spark.catalog.listTables() if "bucketed" in t.name])

## Confirm bucket alignment removes the Exchange

In [ ]:
ob = spark.table("orders_bucketed")
cb = spark.table("customers_bucketed")

print("BUCKETED join plan -- look for NO Exchange feeding the join:")
ob.join(cb, "customer_id").explain()

## Break alignment on purpose: mismatched bucket count falls back silently

In [ ]:
spark.sql("DROP TABLE IF EXISTS customers_bucketed12")
(customers.write.mode("overwrite")
 .bucketBy(12, "customer_id").sortBy("customer_id")   # 12, not 8 -- mismatched
 .saveAsTable("customers_bucketed12"))

print("MISMATCHED bucket count (8 vs 12) -- Exchange is BACK:")
ob.join(spark.table("customers_bucketed12"), "customer_id").explain()
# no error, no warning -- just a normal shuffled join. Always verify with explain().

## Simulate "many runs": time the join 5x, bucketed vs not

This is the lesson's actual payoff — bucketing pays a write-time cost
once and saves the shuffle on every subsequent run.

In [ ]:
import time

def time_join(a, b, label, n=5):
    total = 0.0
    for _ in range(n):
        t0 = time.time()
        a.join(b, "customer_id").agg(F.sum("amount")).collect()
        total += time.time() - t0
    print(f"{label}: {n} runs, avg {total/n:.3f}s/run, total {total:.3f}s")

time_join(orders, customers, "UNBUCKETED (shuffles every run)")
time_join(ob, cb, "BUCKETED   (shuffle paid once, at write time)")

In [ ]:
spark.stop()
shutil.rmtree("spark-warehouse", ignore_errors=True)   # self-clean (4.3 convention)
shutil.rmtree("metastore_db", ignore_errors=True)

## Exercises

1. Reduce the bucket count to 2 and re-run the timing comparison. Does
   bucketing still win? What does 10.3's partition-sizing logic say
   about picking a bucket count too small for your parallelism?
2. Join on a DIFFERENT column than the bucketing key (e.g. bucket on
   `customer_id` but join on `segment`). Confirm via `explain()` that
   bucketing does NOT help — why not, precisely?
3. Add a `.filter()` before the bucketed join. Does the Exchange stay
   gone? What does this tell you about how far bucketing's benefit
   survives being combined with other operations?
4. Bucket `orders` into 8 buckets but `customers` into 8 buckets on a
   DIFFERENT column name that happens to have the same values. Does
   the join skip the shuffle? Revisit the lesson's "same column(s)"
   requirement.
5. Compare bucketing against `broadcast()` (7.2) for this exact
   `orders`/`customers` join — since `customers` is small, would a
   broadcast hint achieve the same Exchange-free join without any
   `saveAsTable` step? When would bucketing still be the better choice
   even though broadcast is available?

In [ ]:
# your exercise code here